# 02 Interface Integration Test (synthetic only, no dataset)

In [ ]:
import sys
sys.path.insert(0, '..')
import numpy as np
from src.data_types import LiDARFrame
from src.feature_adapter import build_perception_from_labeled_points, adapt_perception_to_regions
from src.interface_validator import validate_lidar_frame, validate_perception_result, validate_region_features
from src.stage1_engines import ImportanceEngine, ResolutionEngine, AdaptiveMapper2_5D
print('imports OK')

In [ ]:
# Cell 2: synthetic LiDAR (controlled primitives, NOT real data)
rng = np.random.default_rng(42)
ped = np.column_stack([70 + rng.normal(0, 0.2, 200), 2 + rng.normal(0, 0.2, 200), rng.normal(0.9, 0.1, 200), rng.uniform(0.4, 0.7, 200)])
road = np.column_stack([70 + rng.uniform(-4, 4, 300), -6 + rng.uniform(-2, 2, 300), rng.normal(0, 0.02, 300), rng.uniform(0.2, 0.5, 300)])
pts = np.vstack([ped, road])
labels = np.array(['pedestrian']*len(ped) + ['road']*len(road))
print(pts.shape, 'synthetic Nx4 OK')

In [ ]:
# Cell 3: LiDARFrame
frame = LiDARFrame(frame_id='synth-70m', timestamp=0.0, points=pts)
validate_lidar_frame(frame)
print('LiDARFrame valid:', frame.points.shape)

In [ ]:
# Cell 4: PerceptionResult (synthetic perception values)
conf = np.array([0.85]*len(ped) + [0.95]*len(road))
pr = build_perception_from_labeled_points('synth-70m', pts, labels, conf)
print('PerceptionResult:', pr.points.shape, len(pr.semantic_labels))

In [ ]:
# Cell 5: validate PerceptionResult
validate_perception_result(pr)
print('PerceptionResult valid')

In [ ]:
# Cell 6: PerceptionResult -> RegionFeatures
regions = adapt_perception_to_regions(pr, bin_m=4.0, min_points=15)
print('regions:', len(regions))
for r in regions[:6]:
    print(r.region_id, r.semantic_label, round(r.distance, 1), r.point_count)

In [ ]:
# Cell 7: validate RegionFeatures
for r in regions:
    validate_region_features(r)
print('all RegionFeatures valid')

In [ ]:
# Cell 8: existing Importance Engine (unchanged)
imp = ImportanceEngine()
ix = np.floor(pts[:, 0]/4.0).astype(int)
iy = np.floor(pts[:, 1]/4.0).astype(int)
_, inv = np.unique(np.column_stack([ix, iy]), axis=0, return_inverse=True)
full = [r.to_legacy_region(pts[inv == r.region_id]) for r in regions if (inv == r.region_id).sum() > 0]
scored = [imp.score_region(lr) for lr in full]
for s in scored[:6]:
    print(s.region_id, round(s.base_importance, 3), round(s.safe_importance, 3))

In [ ]:
# Cell 9: existing Resolution Engine (unchanged)
res = ResolutionEngine()
for s in scored:
    m, lvl = res.select(s.safe_importance)
    s.selected_resolution_m, s.resolution_level = m, lvl
for s in scored[:6]:
    print(s.region_id, round(s.safe_importance, 3), s.selected_resolution_m, s.resolution_level)

In [ ]:
# Cell 10: existing Adaptive 2.5D Mapper (unchanged)
mapper = AdaptiveMapper2_5D(imp, res)
mres = mapper.map_regions_legacy(full)
print('cells:', len(mres.cells), 'points:', mres.n_points, 'time_s:', round(mres.elapsed_s, 3))

In [ ]:
# Cell 11: interface proof table
import pandas as pd
rows = [{'region_id': full[i].region_id, 'label': full[i].semantic_class, 'dist_m': round(full[i].distance_m, 1), 'base': round(s.base_importance, 3), 'safe': round(s.safe_importance, 3), 'res_m': s.selected_resolution_m} for i, s in enumerate(scored)]
df = pd.DataFrame(rows)
print(df.to_string(index=False))

In [ ]:
# Cell 12: semantic-vs-distance USP (measured, both near 70 m)
print(df.sort_values('dist_m').to_string(index=False))
print('Result: pedestrian near 70 m gets finer resolution than road near 70 m.')
print('Importance-driven, computed above from synthetic data.')